This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [ ]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [2]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from pathlib import Path
from config import IM_SIZE

# Setup Path
FINN_BIT_WIDTH = 8
onnx_path = f'../onnx/quant_model_{FINN_BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

model = ModelWrapper(onnx_path)

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
tidy_path = f'./tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

showSrc(InferShapes)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


class InferShapes(Transformation):
    """Ensure every tensor in the model has a specified shape (ValueInfo)."""

    def apply(self, model):
        # hide your riches!
        hidden_ops = _hide_finn_ops(model)
        # call regular ONNX shape inference
        model = ModelWrapper(si.infer_shapes(model.model))
        # bring back hidden ops
        _restore_finn_ops(model, hidden_ops)
        return (model, False)



In [3]:
showInNetron(tidy_path)

Serving './tidy_onnx/quant_model_8_bits_tidy.onnx' at http://0.0.0.0:8081


### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [3]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from onnx import helper

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# verify if the sizes of first and last node outputs are correct
first_node_out = model.graph.node[0].output[0]
last_node_out = model.graph.node[-1].output[0]
print(f"\nOutput shape shape of first node ({model.graph.node[0].op_type}): {model.get_tensor_shape(first_node_out)}")
print(f"Output shape shape of last node ({model.graph.node[-1].op_type}): {model.get_tensor_shape(last_node_out)}")

# verify if input and output datatypes are correct
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")
print(f"Output datatype: {model.get_tensor_datatype(last_node_out)}")

# Save the preprocessed model
preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

# Print a human readable representation of the graph
print(helper.printable_graph(model.graph))

/home/jose-vitor/finn-repo/deps/brevitas/src/brevitas/__init__.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packag


Output shape shape of first node (Div): [1, 3, 1024, 2048]
Output shape shape of last node (Conv): [1, 19, 128, 256]

Input datatype: UINT8
Output datatype: FLOAT32
graph main_graph (
  %global_in[FLOAT, 1x3x1024x2048]
) initializers (
  %Sub_0_param0[FLOAT, 1x3x1x1]
  %Div_1_param0[FLOAT, 1x3x1x1]
  %Quant_0_param0[FLOAT, 32x3x3x3]
  %BatchNormalization_0_param0[FLOAT, 32]
  %BatchNormalization_0_param1[FLOAT, 32]
  %BatchNormalization_0_param2[FLOAT, 32]
  %BatchNormalization_0_param3[FLOAT, 32]
  %Quant_1_param0[FLOAT, 32x1x3x3]
  %BatchNormalization_1_param0[FLOAT, 32]
  %BatchNormalization_1_param1[FLOAT, 32]
  %BatchNormalization_1_param2[FLOAT, 32]
  %BatchNormalization_1_param3[FLOAT, 32]
  %Quant_2_param0[FLOAT, 48x32x1x1]
  %BatchNormalization_2_param0[FLOAT, 48]
  %BatchNormalization_2_param1[FLOAT, 48]
  %BatchNormalization_2_param2[FLOAT, 48]
  %BatchNormalization_2_param3[FLOAT, 48]
  %Quant_3_param0[FLOAT, 48x1x3x3]
  %BatchNormalization_3_param0[FLOAT, 48]
  %BatchNorm

In [6]:
showInNetron(preproc_path)

Serving './preproc_onnx/quant_model_8_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [7]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input image dtype:", img_tensor.dtype)
print("Input mask shape:", smnt_tensor.shape)
print("Input mask dtype:", smnt_tensor.dtype)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path="../train_environment/model_weights/quant_params/best_quant_model.pth", strict=False)
brevitas_model.eval();

Input image shape: torch.Size([1, 3, 1024, 2048])
Input image dtype: torch.uint8
Input mask shape: torch.Size([1, 1024, 2048])
Input mask dtype: torch.uint8
Carregando modelo best_quant_model


In [8]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:239: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.cat([

Output shape from Brevitas model: torch.Size([1, 19, 128, 256])
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 83.19%



In [9]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

model = ModelWrapper(preproc_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
qonnx_output_upsampled = F.interpolate(torch.from_numpy(qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
qonnx_output_mask = torch.softmax(qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {qonnx_output.shape}\n"
      f"Output shape after upsampling: {qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


Output shape from QONNX model: (1, 19, 128, 256)
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 80.07%



In [17]:
# check output types
print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (brevitas_output_mask == qonnx_output_mask).sum().item()
total_pixels = brevitas_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Brevitas output dtype: torch.float32
QONNX output dtype: float32

Matching pixels: 1941285/2097152 (92.57%)



### Results diagnostics with IA generated code

In [18]:
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import torch

model = ModelWrapper(preproc_path)

# 1. Ignora falsos positivos de tensores vazios
model.check_all_tensor_shapes_specified = lambda: True
model.set_initializer("", np.array([], dtype=np.float32))

# 2. Executa ONNX salvando TODAS as camadas da rede no dicionário
input_tensor_float = img_tensor.detach().cpu().numpy().astype(np.float32)
input_name = model.graph.input[0].name
output_dict = oxe.execute_onnx(model, {input_name: input_tensor_float}, return_full_exec_context=True)

# 3. O "Espião" no PyTorch (Corrigido para QuantTensor)
pytorch_intermediates = {}
def get_hook(name):
    def hook(m, i, o):
        # Desempacota o QuantTensor de forma segura
        tensor_data = o.value if hasattr(o, 'value') else o
        pytorch_intermediates[name] = tensor_data.detach().cpu().numpy()
    return hook

# Pendura o espião na PRIMEIRA convolução real do modelo
hook_handle = None
for name, module in brevitas_model.named_modules():
    if "Conv2d" in str(type(module)) or "QuantConv2d" in str(type(module)):
        hook_handle = module.register_forward_hook(get_hook("first_conv"))
        print(f"✅ Hook do PyTorch ancorado com sucesso em: {name}")
        break

# Executa o PyTorch
brevitas_model.eval()
_ = brevitas_model(img_tensor)

if hook_handle: 
    hook_handle.remove()

# 4. Comparação Forense Direta (A Autópsia)
print("\n--- RAIO-X: PYTORCH vs ONNX ---")

if "Conv_0_out0" in output_dict:
    pt_conv = pytorch_intermediates["first_conv"]
    onnx_conv = output_dict["Conv_0_out0"]
    
    diff_primeira_conv = np.max(np.abs(pt_conv - onnx_conv))
    print(f"Erro na 1ª Convolução | Erro Máximo Absoluto: {diff_primeira_conv:.6f}")
    
    if diff_primeira_conv > 1e-3:
        print("🚨 ALERTA: O modelo já começa errando! Problema de exportação de pesos ou normalização de entrada (Div/Sub).")
    else:
        print("✅ 1ª Conv bateu perfeitamente! O erro está acontecendo mais adiante.")
else:
    print("Atenção: 'Conv_0_out0' não encontrada no grafo do ONNX.")

# Analisando a degradação do sinal ao longo do ONNX
print("\n--- EVOLUÇÃO DAS ATIVAÇÕES NO ONNX ---")
chokepoints = [
    "Quant_45_out0", # Logo após as operações de normalização (Div/Sub iniciais)
    "Conv_0_out0",   # Saída da primeira Convolução
    "Concat_0_out0", # Fusão do Pyramid Pooling
    "global_out"     # Saída Final (onde vemos os 11.27)
]

for cp in chokepoints:
    if cp in output_dict:
        t = output_dict[cp]
        print(f"Nó {cp:15} | Min: {t.min():>8.4f} | Max: {t.max():>8.4f} | Mean: {t.mean():>8.4f}")

✅ Hook do PyTorch ancorado com sucesso em: learning_to_downsample.conv.conv.0

--- RAIO-X: PYTORCH vs ONNX ---
Erro na 1ª Convolução | Erro Máximo Absoluto: 0.000003
✅ 1ª Conv bateu perfeitamente! O erro está acontecendo mais adiante.

--- EVOLUÇÃO DAS ATIVAÇÕES NO ONNX ---
Nó Quant_45_out0   | Min:  -0.3354 | Max:   0.4953 | Mean:   0.0026
Nó Conv_0_out0     | Min:  -5.5231 | Max:   7.1753 | Mean:   0.1012
Nó Concat_0_out0   | Min:  -2.5599 | Max:   2.7927 | Mean:   0.0459
Nó global_out      | Min: -10.0166 | Max:  16.8007 | Mean:  -0.2048


In [19]:
# 1. O "Espião" Macro (Hookando os 4 módulos principais)
pt_macro_hooks = {}

def get_macro_hook(name):
    def hook(m, i, o):
        # Desempacota o QuantTensor se necessário
        tensor_data = o.value if hasattr(o, 'value') else o
        pt_macro_hooks[name] = tensor_data.detach().cpu().numpy()
    return hook

# Ancorando os espiões no final de cada macro-módulo
hooks = []
hooks.append(brevitas_model.learning_to_downsample.register_forward_hook(get_macro_hook('1_Learning_to_Downsample')))
hooks.append(brevitas_model.global_feature_extractor.register_forward_hook(get_macro_hook('2_Global_Feature_Extractor')))
hooks.append(brevitas_model.feature_fusion.register_forward_hook(get_macro_hook('3_Feature_Fusion')))
hooks.append(brevitas_model.classifier.register_forward_hook(get_macro_hook('4_Classifier')))

# Executa o modelo PyTorch para preencher os espiões
print("Executando modelo PyTorch...")
_ = brevitas_model(img_tensor)

# Remove os hooks para manter a memória limpa
for h in hooks: h.remove()

# 2. O Mapeador de Formas (Procurando a equivalência no ONNX)
print("\n--- MAPEAMENTO DE MACRO-MÓDULOS: PYTORCH vs ONNX ---")

for block_name, pt_tensor in sorted(pt_macro_hooks.items()):
    print(f"\n🔍 Bloco: {block_name} | Shape: {pt_tensor.shape}")
    
    best_match_name = None
    min_error = float('inf')
    
    # Busca em TODOS os nós do ONNX algum que tenha o mesmo shape
    for onnx_name, onnx_tensor in output_dict.items():
        if isinstance(onnx_tensor, np.ndarray) and onnx_tensor.shape == pt_tensor.shape:
            err = np.max(np.abs(pt_tensor - onnx_tensor))
            if err < min_error:
                min_error = err
                best_match_name = onnx_name
                
    if best_match_name is not None:
        print(f"   -> Melhor nó ONNX correspondente: {best_match_name}")
        print(f"   -> Erro Máximo Absoluto: {min_error:.6f}")
        
        if min_error > 1e-3:
            print("   🚨 DIVERGÊNCIA DETECTADA! O erro de precisão nasce neste bloco (ou antes dele)!")
        else:
            print("   ✅ BLOCO PERFEITO! A matemática está idêntica até aqui.")
    else:
        print("   ⚠️ Nenhum tensor com esse shape encontrado no ONNX.")

Executando modelo PyTorch...

--- MAPEAMENTO DE MACRO-MÓDULOS: PYTORCH vs ONNX ---

🔍 Bloco: 1_Learning_to_Downsample | Shape: (1, 64, 128, 256)
   -> Melhor nó ONNX correspondente: Quant_55_out0
   -> Erro Máximo Absoluto: 0.024514
   🚨 DIVERGÊNCIA DETECTADA! O erro de precisão nasce neste bloco (ou antes dele)!

🔍 Bloco: 2_Global_Feature_Extractor | Shape: (1, 128, 32, 64)
   -> Melhor nó ONNX correspondente: Relu_27_out0
   -> Erro Máximo Absoluto: 0.609590
   🚨 DIVERGÊNCIA DETECTADA! O erro de precisão nasce neste bloco (ou antes dele)!

🔍 Bloco: 3_Feature_Fusion | Shape: (1, 128, 128, 256)
   -> Melhor nó ONNX correspondente: Quant_109_out0
   -> Erro Máximo Absoluto: 1.359102
   🚨 DIVERGÊNCIA DETECTADA! O erro de precisão nasce neste bloco (ou antes dele)!

🔍 Bloco: 4_Classifier | Shape: (1, 19, 128, 256)
   -> Melhor nó ONNX correspondente: global_out
   -> Erro Máximo Absoluto: 11.271747
   🚨 DIVERGÊNCIA DETECTADA! O erro de precisão nasce neste bloco (ou antes dele)!


In [20]:
pt_micro_hooks = {}

def get_micro_hook(name):
    def hook(m, i, o):
        tensor_data = o.value if hasattr(o, 'value') else o
        pt_micro_hooks[name] = tensor_data.detach().cpu().numpy()
    return hook

# Ancorando os espiões APENAS dentro do módulo problemático
ltd = brevitas_model.learning_to_downsample
hooks = [
    ltd.conv.register_forward_hook(get_micro_hook('1_LTD_Conv')),
    ltd.dsconv1.register_forward_hook(get_micro_hook('2_LTD_DSConv1')),
    ltd.dsconv2.register_forward_hook(get_micro_hook('3_LTD_DSConv2'))
]

# Executa o modelo PyTorch
print("Rodando modelo no microscópio...")
_ = brevitas_model(img_tensor)

for h in hooks: h.remove()

# Mapeador de Formas Micro
print("\n--- MICRO-RAIO-X: Learning to Downsample ---")
for block_name, pt_tensor in sorted(pt_micro_hooks.items()):
    min_error = float('inf')
    best_match_name = None
    
    for onnx_name, onnx_tensor in output_dict.items():
        if isinstance(onnx_tensor, np.ndarray) and onnx_tensor.shape == pt_tensor.shape:
            err = np.max(np.abs(pt_tensor - onnx_tensor))
            if err < min_error:
                min_error = err
                best_match_name = onnx_name
                
    print(f"🔍 {block_name}")
    if best_match_name:
        print(f"   -> ONNX Correspondente: {best_match_name}")
        print(f"   -> Erro Máximo: {min_error:.6f}")
        if min_error > 1e-3:
            print("   🚨 INÍCIO DO EFEITO BORBOLETA AQUI!")
    else:
        print("   ⚠️ Nó correspondente não encontrado.")

Rodando modelo no microscópio...

--- MICRO-RAIO-X: Learning to Downsample ---
🔍 1_LTD_Conv
   -> ONNX Correspondente: Quant_51_out0
   -> Erro Máximo: 0.018217
   🚨 INÍCIO DO EFEITO BORBOLETA AQUI!
🔍 2_LTD_DSConv1
   -> ONNX Correspondente: Quant_53_out0
   -> Erro Máximo: 0.022063
   🚨 INÍCIO DO EFEITO BORBOLETA AQUI!
🔍 3_LTD_DSConv2
   -> ONNX Correspondente: Quant_55_out0
   -> Erro Máximo: 0.024514
   🚨 INÍCIO DO EFEITO BORBOLETA AQUI!


### Post-Processing

If you have a complete model that should output the final mask, than uncomment 'model = model.transform(InsertTopK(k=1))'. This line adds a final operation on the ONNX equivalent to torch.Tensor.argmax(dim=1).

In [11]:
from qonnx.transformation.insert_topk import InsertTopK

# POST PROC
model = ModelWrapper(preproc_path)

# insert Top-1 node at the end
#model = model.transform(InsertTopK(k=1))
# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# Save the postprocessed model
postproc_path = f'./postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(postproc_path)

In [13]:
showInNetron(postproc_path)

Serving './postproc_onnx/quant_model_8_bits_postproc.onnx' at http://0.0.0.0:8081


### Model streamlining

In [14]:
from finn.transformation.streamline import Streamline
# we can see the list of apllied transformations here : showSrc(Streamline)
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(postproc_path)
# STREAMLINE
model = model.transform(Streamline())

# Save the streamlined model
streamlined_path = f'./streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)
model.save(streamlined_path)

AssertionError: Initializer for conv weights is not set.

In [ ]:
showInNetron(streamlined_path)